# Phase 3 — Exploratory Data Analysis (EDA)
## Patterns of Recorded Civilian Targeting in Gaza

**DS312 · Group 7 · Phase 3 notebook template**

This notebook continues from Phase 2. It first loads and checks the saved Phase 2 CSV outputs, then uses each table for the questions it can answer. The analysis moves from event-level descriptions to group comparisons, weekly patterns, and spatial exploration. Replace prompts with results from the saved project snapshot before submission.

> **Reading path:** project questions → Phase 2 CSVs and their roles → data checks → univariate → bivariate → grouped, temporal, and spatial analysis → hypothesis tests → key insights and Phase 4 plan.

## Project context, audience, and decision

### Purpose
Describe patterns, relationships, and unusual periods in the cleaned Gaza data. Use the evidence to refine the Phase 4 plan for spatial clustering and weekly anomaly detection.

### Audience and decision
The audience is human-rights documentation teams, civilian-protection analysts, investigative journalists, and research coordinators. The findings should help prioritize locations, periods, and incident groups for source verification and contextual investigation.

### Interpretation boundary
This notebook describes **recorded ACLED events**. The project focal-event flag is a reproducible coding rule, not proof of intent, responsibility, or legal classification.

### Narrative sequence
1. Read the Phase 2 outputs and confirm their units, dates, and intended use.
2. Describe individual variables before comparing groups.
3. Compare periods, event types, actors, and severity with explicit denominators.
4. Examine district-week and spatial patterns using the tables prepared for those units.
5. Test two exploratory hypotheses and use the results to refine Phase 4.

## Phase 3 guide requirements and notebook map

The DS312 Final Project Guide (Phase 3) asks for univariate analysis; bivariate and multivariate analysis; at least 8 purposeful, labeled visualizations with 1-2 sentence interpretations; at least 2 statistically tested hypotheses with p-values and plain-language conclusions; pattern spotting; answers or refinements to Phase 1 business questions; modeling implications; and a closing section with 5-8 key insights and selected techniques.

This notebook follows that sequence. Each result should be supported by a table or figure, and narrative should sit between code cells. Correlation is not causation.

## Phase 1 business questions guiding this EDA

Phase 1 established five questions. Phase 2 narrowed the working data to Gaza, so this notebook applies the questions within that scope. Before-and-after comparisons use the cleaned event master; post–October 7 temporal analysis uses the weekly CSVs; spatial exploration uses the land focal-incident CSV.

| ID | Question for Phase 3 | Main Phase 2 output | Phase 4 use |
|---|---|---|---|
| BQ1 | How do recorded focal-event counts and their share of political-violence events differ before and after October 7, 2023 in Gaza? | Cleaned event master | Descriptive baseline and sensitivity questions |
| BQ2 | Which recorded actors and event/sub-event types recur among focal events? | Cleaned event master | Interpretable event profiles; review notes before assigning roles |
| BQ3 | Where are post–October 7 land focal events concentrated, and how does precision affect the view? | Focal-incident table | DBSCAN settings and precision sensitivity |
| BQ4 | Which district-weeks show high or sustained focal activity relative to recent local history? | District-week and Gaza-week tables | Weekly anomaly features and baseline choices |
| BQ5 | Which patterns change under alternative periods, geography, or precision restrictions? | Relevant output table(s) | Robustness checks |

A planned question is not a result. Write each result only after the relevant table and plot have been reviewed.

## Phase 2 outputs: what each CSV represents

Phase 2 saved four CSVs in `data/processed/`. They represent different populations and units. Load them in separate cells so it is clear which file supports each part of the analysis. Run this notebook from the project folder so the relative paths resolve.

| Phase 2 CSV | What one row represents | Use it for | Important scope note |
|---|---|---|---|
| `acled_events_clean.csv` | One Gaza event record, across both periods | Event-level variables, before/after comparisons, actor and event-type profiles, fatalities | Retains Gaza events, including coast-coded records and four records without a district. |
| `acled_gaza_week.csv` | One Gaza-wide calendar week after October 7 | Gaza-wide counts, shares, and temporal context | Zero-event weeks are present; final week may be incomplete; share is undefined with zero political events. |
| `acled_district_week.csv` | One known district and calendar week after October 7 | District comparisons, weekly persistence, heatmaps, lag features | District table excludes four records without `admin2`; use it only for district-level analysis. |
| `acled_focal_incidents.csv` | One post–October 7 land focal incident | Coordinate distribution and precision sensitivity | Coast-coded focal events are excluded for land spatial analysis; repeated coordinates are retained. |

The next cells load each file separately. Read the accompanying note before each cell; it explains why that output exists and what it can support.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")

# Run from the project root (the folder containing data/processed/).
DATA_DIR = Path("data/processed")
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_DIR.resolve()}. Run this notebook from the project root."
    )
print("Reading Phase 2 CSVs from:", DATA_DIR.resolve())

### CSV 1 - Cleaned event master — `acled_events_clean.csv`

**Why this file:** it is the event-level Gaza master across the full download window. Use it for variable distributions, event/actor profiles, and the October 7 descriptive comparison. It is not the spatially restricted table.

**Read/check:** parse the saved event dates and restore precision codes as ordered categories; confirm the date span and inspect a few rows.

In [ ]:
master = pd.read_csv(DATA_DIR / "acled_events_clean.csv", parse_dates=["event_date"])
for col in ("geo_precision", "time_precision"):
    master[col] = pd.Categorical(master[col], categories=[1, 2, 3], ordered=True)

print("Clean event master:", master.shape)
print("Event dates:", master.event_date.min().date(), "to", master.event_date.max().date())
display(master[["event_date", "period", "event_type", "fatalities", "focal_event", "admin2"]].head())

### CSV 2 - Gaza-wide weekly table — `acled_gaza_week.csv`

**Why this file:** Phase 2 aligned Gaza-wide event totals to a weekly calendar, including zero-record weeks. Use it to describe weekly counts and shares after October 7 and to inspect the available historical features.

**Read/check:** parse both week boundaries and review completeness, activity, and lag columns.

In [ ]:
gaza_weekly = pd.read_csv(
    DATA_DIR / "acled_gaza_week.csv",
    parse_dates=["week_start", "week_end"],
)
print("Gaza-week table:", gaza_weekly.shape)
display(gaza_weekly[["week_start", "week_end", "complete_week", "political_events",
                     "focal_events", "focal_share", "previous_week_focal",
                     "previous_four_weeks_focal"]].head())

### CSV 3 - District-week table — `acled_district_week.csv`

**Why this file:** each row is a district-week built from records with a known district. Use it to compare district activity over the same calendar weeks and to explore eligible history. Do not substitute it for the Gaza-wide table.

**Read/check:** parse week boundaries and verify the district and week fields before building grouped views.

In [ ]:
district_weekly = pd.read_csv(
    DATA_DIR / "acled_district_week.csv",
    parse_dates=["week_start", "week_end"],
)
print("District-week table:", district_weekly.shape)
print("Districts:", district_weekly.admin2.nunique(),
      "| weeks:", district_weekly.week_start.nunique())
display(district_weekly[["admin2", "week_start", "complete_week", "political_events",
                         "focal_events", "focal_share", "history_eligible"]].head())

### CSV 4 - Land focal-incident table — `acled_focal_incidents.csv`

**Why this file:** Phase 2 prepared post–October 7 focal events for land-based spatial analysis and added coordinates in radians for later haversine clustering. Use the degree coordinates for maps and the precision fields for sensitivity views.

**Read/check:** parse event dates, restore precision categories, then inspect coordinate ranges and repeated locations.

In [ ]:
incidents = pd.read_csv(
    DATA_DIR / "acled_focal_incidents.csv",
    parse_dates=["event_date"],
)
for col in ("geo_precision", "time_precision"):
    incidents[col] = pd.Categorical(incidents[col], categories=[1, 2, 3], ordered=True)
print("Land focal incidents:", incidents.shape)
print("Distinct coordinate pairs:", len(incidents[["latitude", "longitude"]].drop_duplicates()))
print("Precision 1 records:", incidents.geo_precision_1.sum())
display(incidents[["event_date", "location", "latitude", "longitude",
                   "geo_precision", "geo_precision_1"]].head())

### Loading summary and transition to EDA

The files are now loaded as `master`, `gaza_weekly`, `district_weekly`, and `incidents`. The following audit checks expected fields and key scope facts before charts are made. After that, each EDA section names the table it uses so readers can trace every result back to a Phase 2 output.

## Variable inventory: units, roles, and handling

Use this as the minimum variable contract. Add or remove variables only with a written reason tied to a business question.

| Variable(s) | Type / role in EDA | Use | Handling note |
|---|---|---|---|
| `event_date`, `period`, `month_start`, `week_start` | Date / grouping | Before-after, monthly and weekly trends | Periods differ in duration; compare rates/shares as well as counts. |
| `focal_event` | Boolean outcome for description | Numerator in focal counts and shares | Project-derived rule from targeting flag and exact civilian actor label. |
| `political_violence` | Boolean denominator group | Context and denominator for focal share | Exact token matching was used in Phase 2. |
| `targeting_flagged`, `civilian_targeting` | Boolean / source category | Describe ACLED targeting coding | Blank source value is preserved; derived boolean means recorded flag present. |
| `actor1`, `actor2`, `event_type`, `sub_event_type`, `disorder_type` | Categorical | Actor/event profiles and cross-tabs | Actor labels are high-cardinality; show top groups plus an “Other” display bucket only if documented. |
| `fatalities`, `fatalities_log1p`, `high_fatalities_review` | Count / derived review flag | Severity distribution and comparisons | Retain raw counts; review flags do not mean invalid records. |
| `admin2`, `location` | Geographic categories | District summaries and location descriptions | Four events have missing district; exclude only when district is required. |
| `latitude`, `longitude`, `geo_precision`, `geo_precision_1` | Coordinates / quality | Spatial distribution and sensitivity | Preserve geographic geometry; precision codes are categories, not distances. |
| `time_precision`, `time_precision_1` | Quality category / flag | Sensitivity checks | Do not interpret code values as continuous measurements. |
| `recorded_events`, `political_events`, `focal_events`, `focal_share` | Weekly aggregates | Trends and district-week comparisons | `focal_share` is undefined when denominator is zero. |
| `previous_week_focal`, `previous_four_weeks_focal`, `history_eligible` | Historical features | Explore candidate Phase 4 predictors/baselines | Lagged by Phase 2; use only prior weeks for an alerting feature. |

### Basic integrity review


In [ ]:
# Check the loaded Phase 2 tables before plotting or comparing groups.
required = {
    "master": {"event_date", "period", "event_type", "actor1", "fatalities", "focal_event", "political_violence", "admin2"},
    "gaza_weekly": {"week_start", "week_end", "political_events", "focal_events", "focal_share", "complete_week"},
    "district_weekly": {"admin2", "week_start", "focal_events", "political_events", "focal_share", "history_eligible"},
    "incidents": {"event_date", "location", "latitude", "longitude", "geo_precision", "geo_precision_1"},
}
frames = {"master": master, "gaza_weekly": gaza_weekly,
          "district_weekly": district_weekly, "incidents": incidents}
for name, cols in required.items():
    missing_cols = cols - set(frames[name].columns)
    print(name, "missing expected columns:", sorted(missing_cols))

print("\nEvent-level coverage and missing districts")
print("Missing admin2:", master.admin2.isna().sum())
print("\nFocal-event records by period")
display(master.groupby("period", dropna=False).focal_event.agg(["size", "sum", "mean"]))
print("\nTable shapes:")
for name, frame in frames.items():
    print(f"{name}: {frame.shape}")

## Visualization plan: purposeful figures for the guide

The Phase 3 guide requires **at least 8 purposeful visualizations**. The chart sequence below exceeds that minimum. Keep a figure only if it answers its question; place a 1-2 sentence interpretation directly after every chart.

| Figure | Question | Phase 2 table / variables |
|---|---|---|
| V1 | How are reported fatalities distributed? | `master`: `fatalities`, `fatalities_log1p` |
| V2-V4 | Which event types and actor labels are common? What precision codes are present? | `master`: event categories, actors, precision |
| V5-V8 | How do focal shares, severity, and actor/type patterns compare? | `master`: `period`, `political_violence`, `focal_event`, `event_type`, `fatalities`, actor labels |
| V9-V11 | What changes by week, district, and location? | Gaza-week, district-week, and land focal-incident CSVs |
| V12 | Is there an apparent month-of-year pattern? | `master`: `event_date`, `focal_event` |

Every figure needs a question-led title, labeled axes and legend where relevant, and its own interpretation immediately below.

## Optional reusable plotting helpers

The course guide does not require generic univariate, bivariate, or multivariate EDA functions. It requires purposeful analysis, at least eight interpreted visualizations, two tested hypotheses, and a closing modeling plan. Its reusable-pipeline example appears in the advanced Phase 2 cleaning criteria.

A few small plotting helpers can keep styles consistent. Keep filters, denominators, test choices, and interpretation visible in the analysis cells; each chart still needs its own written interpretation.

In [ ]:
def plot_count(data, column, *, title, order=None, ax=None, normalize=False):
    """Count plot for one categorical variable; normalize=True shows within-column shares."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    counts = data[column].value_counts(dropna=False, normalize=normalize)
    if order is not None:
        counts = counts.reindex(order).dropna()
    counts.sort_values().plot(kind="barh", ax=ax, color="#315A7D")
    ax.set_title(title)
    ax.set_xlabel("Share of records" if normalize else "Number of records")
    ax.set_ylabel(column.replace("_", " ").title())
    return ax


def plot_numeric(data, column, *, title, bins=30, log_x=False, ax=None):
    """Histogram for a numeric variable; choose log_x only when the scale is meaningful."""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    values = data[column].dropna()
    ax.hist(values, bins=bins, color="#4B8A78", edgecolor="white")
    if log_x:
        ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Number of records")
    return ax

## 3.2 Univariate analysis — distributions and frequencies

**Source table:** `master` for event-level fields. Use `gaza_weekly` for the weekly distribution and `incidents` only for spatial-input quality. Begin with the distribution of each variable before asking what differs between groups.

Report counts/shares for categories and robust summaries for skewed counts. Show fatalities both as recorded and in the `log1p` display. Do not interpret precision codes as continuous measurements.

### V1 - How are reported fatalities distributed?

**Source:** cleaned event master. Show raw counts and the log1p display so the long tail does not hide zeros or extreme records.

In [ ]:
# Event-level summaries from the cleaned master.
for col in ["event_type", "geo_precision", "time_precision", "admin2"]:
    print(f"\n{col}")
    display(master[col].value_counts(dropna=False).rename("count").to_frame())

print("Fatality summary")
display(master.fatalities.describe(percentiles=[.5, .75, .9, .95, .99]).to_frame("fatalities"))

# V1 candidate: show the skew on the recorded scale. Add a separate log1p view if useful.
plot_numeric(master, "fatalities", title="How are recorded fatalities distributed across Gaza events?")
plt.tight_layout()
plt.show()

### V1 interpretation - fatalities distribution

**Observed pattern:** [Describe the distribution, counts, center/spread, or dominant categories. Include the relevant sample size.]  
**Why it matters:** [Connect the pattern to a Phase 1 business question and state any limitation.]

### V2 — Which event types appear most often?

**Source:** cleaned event master. This is a frequency view, before comparing event types with focal-event status.

In [ ]:
counts = master.event_type.value_counts(dropna=False).sort_values()
ax = counts.plot(kind="barh", figsize=(8, 4.5), color="#315A7D")
ax.set_title("Which recorded event types appear most often in Gaza?")
ax.set_xlabel("Event records")
ax.set_ylabel("ACLED event type")
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Name the most frequent categories and counts/shares. Note that high frequency alone does not mean high focal-event prevalence.]

### V3 — Which actor labels recur in the event master?

**Source:** cleaned event master. Show recorded labels and missingness; do not infer actor roles from counts alone.

In [ ]:
actor_counts = master.actor1.value_counts().head(12).sort_values()
ax = actor_counts.plot(kind="barh", figsize=(9, 5), color="#4B8A78")
ax.set_title("Which actor1 labels are recorded most often?")
ax.set_xlabel("Event records")
ax.set_ylabel("Recorded actor1 label")
plt.tight_layout()
plt.show()
print("actor2 blank records:", int(master.actor2.isna().sum()))

**Interpretation (1-2 sentences):** [Describe the frequent labels and actor2 missingness. Actor labels require note review before describing roles.]

### V4 — How are geographic precision codes distributed?

**Source:** cleaned event master. This describes location quality before examining maps or clusters.

In [ ]:
precision_counts = master.geo_precision.value_counts(dropna=False).sort_index()
ax = precision_counts.plot(kind="bar", figsize=(6, 4), color="#D4A94E")
ax.set_title("How many event records use each geographic precision code?")
ax.set_xlabel("Geographic precision code")
ax.set_ylabel("Event records")
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Describe the precision mix and the limit it places on location-level conclusions.]

### V5 - Did focal-event activity differ before and after October 7?

**Source:** cleaned event master. Period lengths differ, so compare focal events per seven calendar days and focal share among political-violence records.

In [ ]:
cutoff = pd.Timestamp("2023-10-07")
start, end = master.event_date.min(), master.event_date.max()
days = {"Before Oct 7, 2023": (cutoff - start).days,
        "Oct 7, 2023 onward": (end - cutoff).days + 1}
period_summary = master.groupby("period").agg(
    focal_events=("focal_event", "sum"), political_events=("political_violence", "sum"))
period_summary["calendar_days"] = period_summary.index.map(days)
period_summary["focal_events_per_7_days"] = period_summary.focal_events * 7 / period_summary.calendar_days
period_summary["focal_share"] = period_summary.focal_events / period_summary.political_events.replace(0, np.nan)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
period_summary.focal_events_per_7_days.plot(kind="bar", ax=axes[0], color="#8D3B42")
axes[0].set_title("How did focal-event frequency per week differ by period?")
axes[0].set_xlabel("Period"); axes[0].set_ylabel("Focal events per 7 days")
period_summary.focal_share.plot(kind="bar", ax=axes[1], color="#315A7D")
axes[1].set_title("What share of political-violence records were focal?")
axes[1].set_xlabel("Period"); axes[1].set_ylabel("Focal / political-violence records")
plt.tight_layout(); plt.show()
display(period_summary)

**Interpretation (1-2 sentences):** [Compare standardized rate and share with numerator/denominator. Note sparse earlier Gaza focal counts; do not imply the cutoff caused the difference.]

## 3.3 Bivariate and multivariate analysis — relationships and group comparisons

**Source table:** `master` unless the comparison is explicitly weekly or spatial. Start with Phase 1 questions: before/after period differences, event-type composition, actor/event profiles, severity, and precision. Use counts and rates together when group sizes differ.

For every share, state the denominator. For actor comparisons, describe recorded labels and consult notes before assigning actor roles. These are exploratory comparisons; an association is not a causal effect.

### V6 - Which event types have the highest focal-event share?

**Source:** cleaned event master. Use political-violence records as the within-type denominator and display its numerator/denominator table.

In [ ]:
# Example: event-type denominators and focal-event counts/shares.
event_type_summary = master.groupby("event_type", dropna=False).agg(
    events=("event_id_cnty", "size"),
    political_events=("political_violence", "sum"),
    focal_events=("focal_event", "sum"),
)
event_type_summary["focal_share_of_all_events"] = event_type_summary.focal_events / event_type_summary.events
event_type_summary["focal_share_of_political_events"] = (
    event_type_summary.focal_events / event_type_summary.political_events.replace(0, np.nan)
)
display(event_type_summary.sort_values("focal_events", ascending=False))

ax = event_type_summary.sort_values("focal_share_of_political_events")["focal_share_of_political_events"].plot(
    kind="barh", figsize=(9, 5), color="#315A7D"
)
ax.set_title("Within each event type, what share of political-violence records are focal events?")
ax.set_xlabel("Focal events / political-violence events")
ax.set_ylabel("Event type")
plt.tight_layout()
plt.show()

# Interpretation: [Which event types have the largest focal shares? How do counts and denominators affect the comparison?]

### V7 - How do reported fatalities vary by focal status?

**Source:** political-violence records from the cleaned event master. `log1p` makes the long-tailed distribution readable; retain the raw totals for summaries.

In [ ]:
political = master.loc[master.political_violence].copy()
ax = sns.boxplot(data=political, x="focal_event", y="fatalities_log1p", color="#78A99A")
ax.set_title("How does log1p reported fatalities vary by focal status?")
ax.set_xlabel("Project focal-event flag")
ax.set_ylabel("log1p(reported event fatalities)")
plt.tight_layout()
plt.show()
display(political.groupby("focal_event").fatalities.agg(["count", "median", "mean", "max"]))

**Interpretation (1-2 sentences):** [Compare median, spread, and sample size. Reported event fatalities do not automatically identify civilian fatalities.]

### V8 - Which actor labels recur with which event types among focal events?

**Source:** focal-event subset of the cleaned event master. The heatmap shows recorded co-occurrence, not verified actor roles.

In [ ]:
focal = master.loc[master.focal_event].copy()
top_actors = focal.actor1.value_counts().head(8).index
actor_type = pd.crosstab(focal.loc[focal.actor1.isin(top_actors), "actor1"],
                         focal.loc[focal.actor1.isin(top_actors), "event_type"])
plt.figure(figsize=(10, 5))
sns.heatmap(actor_type, annot=True, fmt="g", cmap="crest", cbar_kws={"label": "Focal event records"})
plt.title("Among focal events, which actor1 labels recur with each event type?")
plt.xlabel("Recorded event type")
plt.ylabel("Recorded actor1 label")
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Describe common coded combinations, explain the top-actor selection rule, and identify notes for review.]

**Interpretation (1-2 sentences):**

**Observed pattern:** [Identify event types with high counts and/or high within-type focal shares. Quote their denominators.]  
**Why it matters:** [Explain how this helps answer BQ2 and what should be reviewed in event notes.]  
**Limit:** [State how category size, coding, or missingness affects the comparison.]

## 3.3 continued - grouped, temporal, and spatial relationships

This section changes the unit of analysis. `gaza_weekly` is one Gaza-wide week; `district_weekly` is one known district-week; `incidents` is one post–October 7 land focal incident. Keep each figure tied to its table and unit.

For temporal figures, distinguish event counts from shares and mark the incomplete final week. For district comparisons, show denominators and keep unknown-district records in Gaza-wide totals. For spatial figures, compare precision subsets on the same map extent and describe recorded concentrations rather than exact attack points.

### V9 - How does Gaza-wide focal activity change by week?

**Source:** `acled_gaza_week.csv`. Counts and shares use different scales; the final incomplete week must be identified.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(gaza_weekly.week_start, gaza_weekly.focal_events, color="#8D3B42")
axes[0].set_title("How many focal events were recorded each Gaza week?")
axes[0].set_ylabel("Focal events")
axes[1].plot(gaza_weekly.week_start, gaza_weekly.focal_share, color="#315A7D")
axes[1].set_title("What share of political-violence records were focal each week?")
axes[1].set_ylabel("Focal share")
axes[1].set_xlabel("Week start")
for ax in axes:
    for date in gaza_weekly.loc[~gaza_weekly.complete_week, "week_start"]:
        ax.axvline(date, color="#D4A94E", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Describe sustained changes/spikes; identify any zero-denominator weeks and the incomplete week. Zero recorded events do not prove no harm.]

### V10 - Which districts show high focal-event weeks?

**Source:** `acled_district_week.csv`. This heatmap uses district-week counts and excludes records with missing district.

In [ ]:
heat = district_weekly.pivot(index="admin2", columns="week_start", values="focal_events")
plt.figure(figsize=(14, max(4, .45 * len(heat))))
sns.heatmap(heat, cmap="mako", cbar_kws={"label": "Recorded focal events"})
plt.title("Which known Gaza districts have high focal-event weeks, and when?")
plt.xlabel("Week start")
plt.ylabel("District (admin2)")
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Identify concentration or persistence if visible. Distinguish high counts from high shares and state the missing-district limitation.]

### V11 - Where are post-October 7 land focal incidents recorded?

**Source:** `acled_focal_incidents.csv`. Compare location precision on the same coordinate extent.

In [ ]:
plt.figure(figsize=(7, 8))
sns.scatterplot(data=incidents, x="longitude", y="latitude", hue="geo_precision_1",
                palette={True: "#8D3B42", False: "#315A7D"}, alpha=.45, s=18)
plt.title("Where are land focal incidents recorded, by geographic precision?")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.legend(title="Precision code 1")
plt.tight_layout()
plt.show()

**Interpretation (1-2 sentences):** [Describe visible concentrations and whether they appear in the precision-1 subset. Repeated coordinates and uncertainty limit point-level interpretation.]

## 3.4 Hypothesis testing

Set the hypothesis before looking at the test result. Use **at least two** tests aligned with the business questions. Report the test statistic, p-value, effect size, assumptions, and a plain-language conclusion. Treat tests as exploratory because events can be related by time and place.

### Hypothesis A — before/after focal share
- **Question:** Did the share of political-violence event records classified as focal differ between the two periods?
- **H0:** Period and focal-event status are independent among political-violence records.
- **H1:** Period and focal-event status are associated.
- **Table:** Rows = period; columns = focal yes/no; restrict denominator to `political_violence == True`.
- **Candidate test:** Chi-square independence if expected counts are adequate; exact/Monte Carlo alternative if not. Report Cramér’s V.
- **Caution:** Periods have unequal duration, and earlier focal counts may be sparse. Show counts and per-week context; do not imply period caused the difference.

### Hypothesis B — event type and focal status
- **Question:** Does focal-event prevalence vary by recorded event type?
- **H0:** Event type and focal-event status are independent among political-violence records.
- **H1:** At least one event type has a different focal-event prevalence.
- **Table:** Rows = event type; columns = focal yes/no; denominator restricted to political-violence records.
- **Candidate test:** Chi-square independence; use Monte Carlo resampling if expected counts are small. Report Cramér’s V and within-type rates.
- **Caution:** The test is an association in this export. It does not establish event-type effects or causal relationships.

### Test report table — fill after running

| Hypothesis | Test and assumptions checked | Statistic | p-value | Effect size | Plain-language conclusion / limitation |
|---|---|---:|---:|---:|---|
| A: period × focal status | [Fill] | [Fill] | [Fill] | [Fill] | [Fill] |
| B: event type × focal status | [Fill] | [Fill] | [Fill] | [Fill] | [Fill] |

Use a preselected significance threshold (for example, α = 0.05) and disclose multiple exploratory comparisons if you run additional tests.

In [ ]:
# Hypothesis 1: period x focal status among political-violence records
from scipy.stats import chi2_contingency, MonteCarloMethod

period_table = pd.crosstab(political.period, political.focal_event)
period_asymptotic = chi2_contingency(period_table, correction=False)
period_expected = pd.DataFrame(period_asymptotic.expected_freq,
                               index=period_table.index, columns=period_table.columns)
print("Hypothesis 1 - observed counts")
display(period_table)
print("Expected counts")
display(period_expected)
if (period_expected.to_numpy() < 5).any():
    period_result = chi2_contingency(
        period_table, correction=False,
        method=MonteCarloMethod(n_resamples=19_999, rng=np.random.default_rng(42)),
    )
    period_method = "Chi-square statistic with Monte Carlo p-value"
else:
    period_result = period_asymptotic
    period_method = "Pearson chi-square"
period_v = np.sqrt(period_result.statistic / (period_table.to_numpy().sum() * min(period_table.shape[0]-1, period_table.shape[1]-1)))
print("Method:", period_method)
print(f"Statistic={period_result.statistic:.3f}; p-value={period_result.pvalue:.6g}; Cramer's V={period_v:.3f}")

# Hypothesis 2: event type x focal status among political-violence records
event_type_table = pd.crosstab(political.event_type, political.focal_event)
event_type_asymptotic = chi2_contingency(event_type_table, correction=False)
event_type_expected = pd.DataFrame(event_type_asymptotic.expected_freq,
                                   index=event_type_table.index, columns=event_type_table.columns)
print("\nHypothesis 2 - observed counts")
display(event_type_table)
print("Expected counts")
display(event_type_expected)
if (event_type_expected.to_numpy() < 5).any():
    event_type_result = chi2_contingency(
        event_type_table, correction=False,
        method=MonteCarloMethod(n_resamples=19_999, rng=np.random.default_rng(42)),
    )
    event_type_method = "Chi-square statistic with Monte Carlo p-value"
else:
    event_type_result = event_type_asymptotic
    event_type_method = "Pearson chi-square"
event_type_v = np.sqrt(event_type_result.statistic / (event_type_table.to_numpy().sum() * min(event_type_table.shape[0]-1, event_type_table.shape[1]-1)))
print("Method:", event_type_method)
print(f"Statistic={event_type_result.statistic:.3f}; p-value={event_type_result.pvalue:.6g}; Cramer's V={event_type_v:.3f}")

**Hypothesis 1 conclusion (plain language):** [Name the test, statistic, p-value, effect size, period shares, and uncertainty. State whether the sample provides evidence of an association; do not claim the date cutoff caused it.]

**Hypothesis 2 conclusion (plain language):** [Name the test, statistic, p-value, effect size, within-type shares, and any sparse-category caveat. State what the association does and does not establish.]

## 3.5 Pattern spotting: segments, trends, seasonality, unusual values, and imbalance

The guide asks the team to look beyond individual plots for meaningful patterns. Use this section to identify what deserves follow-up, while distinguishing event patterns from reporting or coding patterns. With a limited historical window, month-of-year differences are exploratory and should not be described as a stable seasonal cycle.

### V12 - Is there a repeated month-of-year pattern in recorded focal events?

**Table:** cleaned event master. This view compares calendar months across years; partial coverage and changing reporting can affect the pattern.

In [ ]:
seasonal = master.assign(year=master.event_date.dt.year, calendar_month=master.event_date.dt.month)
seasonal_counts = seasonal.pivot_table(index="year", columns="calendar_month", values="focal_event", aggfunc="sum", fill_value=0)
plt.figure(figsize=(11, 4))
sns.heatmap(seasonal_counts, cmap="rocket", cbar_kws={"label": "Recorded focal events"})
plt.title("How do recorded focal-event counts vary by calendar month and year?")
plt.xlabel("Calendar month")
plt.ylabel("Year")
plt.tight_layout()
plt.show()
display(seasonal_counts)

**Interpretation (1-2 sentences):** [Describe any repeated month pattern, if present. Note that fewer than four annual cycles, partial coverage, and changing conflict/reporting conditions limit seasonal claims.]

### Suspicious values and class imbalance review

Use the Phase 2 `high_fatalities_review` flag as a source-review queue, not a deletion rule. Report the focal/non-focal class balance before Phase 4; imbalance affects how anomaly results and evaluation measures should be interpreted.

In [ ]:
print("High-fatality review flags:")
display(master.high_fatalities_review.value_counts(dropna=False).rename("count").to_frame())
print("Focal-event class balance:")
display(master.focal_event.value_counts(dropna=False).rename("count").to_frame())
display(master.focal_event.value_counts(normalize=True, dropna=False).rename("share").to_frame())
print("Highest reported fatality events for source review:")
display(master.nlargest(10, "fatalities")[["event_id_cnty", "event_date", "location", "event_type", "fatalities", "source"]])

**Pattern summary:** [Which segments, trends, repeated timing patterns, unusual records, and class imbalance matter for the research questions? State what may instead reflect coverage or source coding.]

## 3.6 Answer or refine the Phase 1 business questions

Use the figures and tests to answer each question in words. If the available table cannot answer it, say what additional evidence is needed.

| Business question | Evidence to cite | Answer / refinement | Remaining uncertainty |
|---|---|---|---|
| BQ1: before/after counts and shares | V-period comparison; Hypothesis 1 | [Write answer] | [Unequal duration, sparse baseline, coverage] |
| BQ2: recurring actors and event types | Event-type and actor charts/cross-tabs; Hypothesis 2 | [Write answer; list notes for review] | [Coding and missingness] |
| BQ3: geographic concentrations | Spatial chart and precision comparison | [Write answer] | [Precision, repeated coordinates, coast restriction] |
| BQ4: unusual district-weeks | Weekly trend and district-week heatmap | [Write answer] | [History eligibility, reporting, zero counts] |
| BQ5: sensitivity | Period, precision, and baseline comparisons | [State which patterns appear stable] | [What remains untested] |

## 3.7 Modeling implications from the EDA

The guide asks which features look useful, which techniques look promising, and which data issues matter. Base choices on evidence above. Review class imbalance and multicollinearity/redundancy; for example, counts, shares, and rolling counts can encode overlapping information.

| Candidate | EDA evidence | Implication for Phase 4 | Issue to handle |
|---|---|---|---|
| DBSCAN spatial clustering | [Spatial concentration and precision sensitivity] | [Retain/refine; justify distance and parameter ranges] | Approximate coordinates, repeated locations, coast scope. |
| Weekly anomaly detection | [Spikes, persistence, eligible history] | [Retain/refine trailing median/MAD and Isolation Forest comparison] | Chronological validation, sparse history, incomplete week, reporting changes. |
| Candidate features | [Group comparisons and weekly correlations] | [List features available before the assessed week] | Avoid leakage from same-week outcomes; check redundant predictors. |
| Class imbalance | [Focal prevalence] | [State its effect on alert interpretation/metrics] | Avoid accuracy-only evaluation; use metrics suited to the task. |

## 3.8 Key insights and chosen modeling plan

### Key insights (final submission: 5-8 evidence-backed bullets)
1. **[Insight]** [Metric/figure/table and which business question it answers.]  
2. **[Insight]** [Evidence and limitation.]  
3. **[Insight]** [Evidence and limitation.]  
4. **[Insight]** [Evidence and limitation.]  
5. **[Insight]** [Evidence and limitation.]  
6. **[Optional insight]** [...]  
7. **[Optional insight]** [...]  
8. **[Optional insight]** [...]

### Chosen techniques and modeling plan

| Proposed technique | EDA evidence that supports it | Inputs / target | Baseline or comparison | Risks / sensitivity checks |
|---|---|---|---|---|
| DBSCAN spatial clustering | [Concentration, repeated locations, precision pattern] | Land focal-event coordinates in radians; Haversine distance | District-level count summaries | Vary epsilon/min_samples; compare precision-1 subset; interpret clusters as recorded concentrations. |
| Weekly anomaly detection | [Visible spikes/persistence and historical distribution] | District-week focal counts plus eligible prior-week history | Trailing median/MAD rule | Chronological evaluation; incomplete weeks and sparse history; compare with Isolation Forest if retained. |

Only retain techniques supported by the EDA and Phase 1 plan. Record any revisions to the modeling questions and why the revision follows from evidence.

## Limitations, reproducibility, and references

### Limitations to report
- ACLED is a recorded-event dataset and may underrepresent events; reporting coverage can change across time and place.
- Fatality totals are event-level reported values and do not automatically identify civilian fatalities.
- Actor, targeting, location, and precision fields reflect the source’s coding and may be incomplete or approximate.
- Observations may be related across time and geography, limiting simple independence assumptions in statistical tests.
- Descriptive differences and statistical associations do not establish causation, intent, or legal classification.

### Reproducibility checklist
- [ ] Phase 2 exports are loaded from `data/processed` and source snapshot is identified.
- [ ] Date parsing and precision category restoration are explicit.
- [ ] Every subset/filter is described, including coast exclusion and district missingness.
- [ ] Randomized tests include a fixed seed and number of resamples.
- [ ] Notebook runs from top to bottom and all charts/tables reconcile to their analysis table.

### References
- Group 7, *Phase 1 — Dataset Selection and Business Problem*.
- Group 7, *Phase 2 — Data Preprocessing*.
- *DS312 Final Project Guide*, Phase 3: Exploratory Data Analysis (EDA).
- ACLED, *Codebook* and the documented event-level export snapshot.